In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
zip_path = "/content/drive/MyDrive/VIP_Midterm/CellMob_AUB.zip"
print("File exists:",os.path.exists(zip_path))

File exists: True


In [ ]:
import os
import zipfile
zip_path="/content/drive/MyDrive/VIP_Midterm/CellMob_AUB.zip"
extract_path = "/content/CellMob_AUB"
os.makedirs(extract_path,exist_ok=True)
with zipfile.ZipFile(zip_path,'r') as zip_ref:
      zip_ref.extractall(extract_path)
print("Dataset extracted succcessfully.")

Dataset extracted succcessfully.


In [ ]:
import os
for root,dirs,files in os.walk("/content/CellMob_AUB"):
    print("Folder:",root)
    print("Files:",files[:10])
    print("-"*40)

Folder: /content/CellMob_AUB
Files: []
----------------------------------------
Folder: /content/CellMob_AUB/CellMob_AUB
Files: ['CellMob.zip']
----------------------------------------
Folder: /content/CellMob_AUB/CellMob_AUB/CellMob
Files: []
----------------------------------------
Folder: /content/CellMob_AUB/CellMob_AUB/CellMob/car_mekkah
Files: ['car_mekkah_064.csv', 'car_mekkah_065.csv', 'car_mekkah_062.csv', 'car_mekkah_061.csv', 'car_mekkah_058.csv', 'car_mekkah_060.csv', 'car_mekkah_063.csv', 'car_mekkah_066.csv', 'car_mekkah_059.csv']
----------------------------------------
Folder: /content/CellMob_AUB/CellMob_AUB/CellMob/walk_kz
Files: ['walk_kz_142.csv']
----------------------------------------
Folder: /content/CellMob_AUB/CellMob_AUB/CellMob/bus_colored_kaust
Files: ['bus_colored_kaust_006.csv', 'bus_colored_kaust_002.csv', 'bus_colored_kaust_004.csv', 'bus_colored_kaust_005.csv', 'bus_colored_kaust_001.csv', 'bus_colored_kaust_007.csv', 'bus_colored_kaust_003.csv']
-----

In [ ]:
import pandas as pd
sample_file = "/content/CellMob_AUB/CellMob_AUB/CellMob/car_mekkah/car_mekkah_064.csv"
df_sample = pd.read_csv(sample_file)
print(df_sample.head())
print("\nColumns:\n",df_sample.columns.tolist())
print("\nShape:",df_sample.shape)

           Time  Longitude  Latitude  Velocity  RSRP/antenna port - 1  \
0  12:32:17.422        NaN       NaN       NaN                    NaN   
1  12:32:17.422        NaN       NaN       NaN                    NaN   
2  12:32:17.422        NaN       NaN       NaN                    NaN   
3  12:32:17.422        NaN       NaN       NaN                  -88.2   
4  12:32:17.422        NaN       NaN       NaN                    NaN   

   RSRP/antenna port - 2  RSRP/antenna port - 3  RSRP/antenna port - 4  \
0                    NaN                    NaN                    NaN   
1                    NaN                    NaN                    NaN   
2                    NaN                    NaN                    NaN   
3                  -74.9                  -94.8                  -76.7   
4                    NaN                    NaN                    NaN   

   RSRP/antenna port - 5  RSRP/antenna port - 6  ...  \
0                    NaN                    NaN  ...   
1   

/tmp/ipykernel_217/463190915.py:3: DtypeWarning: Columns (31,32,33,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sample = pd.read_csv(sample_file)


In [ ]:
import os
import pandas as pd
import numpy as np

base_dirs = [
    "/content/CellMob_AUB/CellMob_AUB/CellMob",
    "/content/CellMob_AUB/CellMob_AUB/KAUST_additional_modes"
]

samples = []

for base_dir in base_dirs:
   for label in sorted(os.listdir(base_dir)):
        label_path = os.path.join(base_dir, label)
        if not os.path.isdir(label_path):
            continue

        for file_name in sorted(os.listdir(label_path)):
            if not file_name.endswith(".csv"):
                continue

            file_path = os.path.join(label_path, file_name)

            try:
                df = pd.read_csv(file_path, low_memory=False)

                # keep numeric columns only
                numeric_df = df.select_dtypes(include=[np.number])

                # skip empty numeric files
                if numeric_df.shape[1] == 0:
                    continue

                feature_dict = {}

                # summarize each numeric column
                for col in numeric_df.columns:
                    feature_dict[f"{col}_mean"] = numeric_df[col].mean()
                    feature_dict[f"{col}_std"] = numeric_df[col].std()
                    feature_dict[f"{col}_min"] = numeric_df[col].min()
                    feature_dict[f"{col}_max"] = numeric_df[col].max()

                feature_dict["label"] = label
                feature_dict["file_name"] = file_name

                samples.append(feature_dict)

            except Exception as e:
                print(f"Error reading {file_path}: {e}")

dataset = pd.DataFrame(samples)

print("Dataset shape:", dataset.shape)
print(dataset.head())
print("\nLabels:")
print(dataset["label"].value_counts())

Dataset shape: (228, 350)
   Longitude_mean  Longitude_std  Longitude_min  Longitude_max  Latitude_mean  \
0       39.799754       0.012452      39.782276      39.835209      21.383015   
1       39.848924       0.016716      39.822090      39.875378      21.389416   
2       39.817251       0.005935      39.806850      39.834599      21.415548   
3       39.836326       0.019519      39.806778      39.863453      21.448804   
4       39.836056       0.022195      39.805950      39.877983      21.422658   

   Latitude_std  Latitude_min  Latitude_max  Velocity_mean  Velocity_std  ...  \
0      0.014605     21.361115     21.414467      45.364884     17.383482  ...   
1      0.020108     21.360163     21.420784      41.841375     25.843796  ...   
2      0.011791     21.397366     21.435524      23.943654     24.303855  ...   
3      0.011841     21.432087     21.477705      38.646013     20.717936  ...   
4      0.019888     21.380718     21.447947      34.379630     19.923646  ...   



In [ ]:
X = dataset_filtered.drop(columns=["label", "file_name"], errors="ignore")
y = dataset_filtered["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (225, 348)
y shape: (225,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (180, 348)
Test shape: (45, 348)


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

imputer = SimpleImputer(strategy="mean")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train_imputed, y_train)

print("Model trained successfully.")

NameError: name 'X_train' is not defined

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_imputed)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

NameError: name 'model' is not defined

In [ ]:
print("Final Accuracy:", acc)
print("\nClass counts:\n")
print(dataset_filtered["label"].value_counts())

NameError: name 'acc' is not defined

In [ ]:
print("Centralized baseline accuracy:", acc)
print("\nClass counts:")
print(dataset_filtered["label"].value_counts())

NameError: name 'acc' is not defined

In [ ]:
print("Dataset shape:", dataset.shape)
print("Filtered dataset exists:", "dataset_filtered" in globals())
if "dataset_filtered" in globals():
    print("Filtered shape:", dataset_filtered.shape)

Dataset shape: (228, 350)
Filtered dataset exists: False


In [ ]:
import numpy as np

print("Total NaNs in dataset:", dataset.isna().sum().sum())
print("Rows in dataset:", len(dataset))
print("Columns in dataset:", len(dataset.columns))

Total NaNs in dataset: 34990
Rows in dataset: 228
Columns in dataset: 350


In [ ]:
# show columns that still have many real values
notna_counts = dataset.notna().sum().sort_values(ascending=False)
print(notna_counts.head(20))

file_name                                     228
label                                         228
Channel number (LTE pcell)_std                228
Channel number (LTE pcell)_mean               228
RSRP/antenna port - 2_max                     228
RSRP/antenna port - 2_min                     228
RSRP/antenna port - 2_std                     228
RSRP/antenna port - 2_mean                    228
Rank indication - 3_max                       228
Rank indication - 3_min                       228
Rank indication - 3_std                       228
Rank indication - 3_mean                      228
Rank indication - 2_max                       228
Rank indication - 2_min                       228
Channel number (LTE pcell)_min                228
Channel number (LTE pcell)_max                228
E-UTRAN carrier RSSI/antenna port - 1_max     228
E-UTRAN carrier RSSI/antenna port - 1_min     228
E-UTRAN carrier RSSI/antenna port - 1_std     228
E-UTRAN carrier RSSI/antenna port - 1_mean    228


In [ ]:
import numpy as np

print("NaNs in X_train_imputed:", np.isnan(X_train_imputed).sum())
print("NaNs in X_test_imputed:", np.isnan(X_test_imputed).sum())

NameError: name 'X_train_imputed' is not defined

In [ ]:
label_counts = dataset["label"].value_counts()
valid_labels = label_counts[label_counts >= 3].index

dataset_filtered = dataset[dataset["label"].isin(valid_labels)].copy()

print("Filtered dataset shape:", dataset_filtered.shape)
print("\nRemaining labels:")
print(dataset_filtered["label"].value_counts())

Filtered dataset shape: (225, 350)

Remaining labels:
label
walk_kaust           69
scooter              29
walk_mekkah          18
bus_mekkah           17
bike                 17
car_jeddah           13
motorcycle           10
car_mekkah            9
jog                   8
car_kz                8
car_kaust             8
bus_colored_kaust     7
bus_jeddah            4
walk_jeddah           4
bus_ondemand          4
Name: count, dtype: int64


In [ ]:
X = dataset_filtered.drop(columns=["label", "file_name"], errors="ignore")
y = dataset_filtered["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (225, 348)
y shape: (225,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (180, 348)
Test shape: (45, 348)


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import numpy as np

all_nan_cols = X_train.columns[X_train.isna().all()]
X_train = X_train.drop(columns=all_nan_cols)
X_test = X_test.drop(columns=all_nan_cols)

print("Dropped all-NaN columns:", len(all_nan_cols))

imputer = SimpleImputer(strategy="mean")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("NaNs in X_train_imputed:", np.isnan(X_train_imputed).sum())
print("NaNs in X_test_imputed:", np.isnan(X_test_imputed).sum())

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train_imputed, y_train)
print("Model trained successfully.")

Dropped all-NaN columns: 0
NaNs in X_train_imputed: 0
NaNs in X_test_imputed: 0
Model trained successfully.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_imputed)
acc = accuracy_score(y_test, y_pred)

print("Accuracy:", acc)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.8888888888888888

Classification Report:

                   precision    recall  f1-score   support

             bike       0.67      0.67      0.67         3
bus_colored_kaust       0.00      0.00      0.00         1
       bus_jeddah       1.00      1.00      1.00         1
       bus_mekkah       0.75      1.00      0.86         3
     bus_ondemand       0.50      1.00      0.67         1
       car_jeddah       1.00      1.00      1.00         2
        car_kaust       1.00      1.00      1.00         1
           car_kz       1.00      1.00      1.00         2
       car_mekkah       1.00      0.50      0.67         2
              jog       1.00      0.50      0.67         2
       motorcycle       1.00      1.00      1.00         2
          scooter       0.86      1.00      0.92         6
      walk_jeddah       1.00      1.00      1.00         1
       walk_kaust       0.93      1.00      0.97        14
      walk_mekkah       1.00      0.75      0.86         4



In [ ]:
print("Centralized baseline accuracy:", acc)
print("\nClass counts:")
print(dataset_filtered["label"].value_counts())

Centralized baseline accuracy: 0.8888888888888888

Class counts:
label
walk_kaust           69
scooter              29
walk_mekkah          18
bus_mekkah           17
bike                 17
car_jeddah           13
motorcycle           10
car_mekkah            9
jog                   8
car_kz                8
car_kaust             8
bus_colored_kaust     7
bus_jeddah            4
walk_jeddah           4
bus_ondemand          4
Name: count, dtype: int64


In [ ]:
print("Centralized baseline accuracy: 0.8889")

Centralized baseline accuracy: 0.8889
